# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saarthak181/FlyRankAI_Week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [40]:
import os
import duckdb
from google.colab import userdata

# Get Hugging Face READ token from Colab Secrets.
# Secret name must be exactly: HF_TOKEN
HF_TOKEN = userdata.get("HF_TOKEN")

print("✓ HF_TOKEN loaded")

# Connect DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Keep the token out of SQL text.
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

# February = feature window
FEB = f"{FACT}/month=2026-02/*.parquet"

# March = label/outcome window
MAR = f"{FACT}/month=2026-03/*.parquet"

print("✓ DuckDB connected")
print("✓ FlyRank warehouse configured")
print("✓ Feature window: February 2026")
print("✓ Label window: March 2026")

✓ HF_TOKEN loaded
✓ DuckDB connected
✓ FlyRank warehouse configured
✓ Feature window: February 2026
✓ Label window: March 2026


In [41]:
schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{FEB}')
""").fetchdf()

display(schema)

print("Number of columns:", len(schema))

print("\nColumn names:")
for col in schema["column_name"]:
    print("-", col)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Number of columns: 31

Column names:
- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other
- scroll_events
- month


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis + time window

The raw warehouse fact table has one row per report date × client × content item.

For my analysis, I create one decision-level observation for each client_hash_id × content_hash_id pair using a February 2026 feature window and a March 2026 outcome window.

The decision cutoff is 28 February 2026. All features must be knowable by that cutoff. March 2026 is used only to measure the subsequent outcome.

My lane is Refresh / Content Opportunity Scoring. The goal is to identify content items that may be worth reviewing for a possible content improvement. The output is directional decision-support, not proof that a refresh will improve performance.

In [42]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{FEB}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchdf()

display(grain_check)

if len(grain_check) == 0:
    print("✓ No duplicate report_date × client_hash_id × content_hash_id rows found.")
    print("✓ Raw fact-table grain is verified.")
else:
    print("⚠ Duplicate rows found. Inspect the table above.")

,report_date,client_hash_id,content_hash_id,row_count


✓ No duplicate report_date × client_hash_id × content_hash_id rows found.
✓ Raw fact-table grain is verified.


In [43]:
feb_window = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{FEB}')
""").fetchdf()

display(feb_window)

,row_count,first_date,last_date
0,7355108,2026-02-01,2026-02-28


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features

My features come only from the February 2026 feature window, before the 28 February decision cutoff. I will use historical search-performance measurements and derived measurements calculated only from this window.

The first feature frame contains five features:

imp_feb — February search impressions.
clk_feb — February search clicks.
avg_position_feb — impression-weighted average search position in February.
days_gsc_feb — number of February days with measured GSC data.
ctr_feb — February CTR calculated from February clicks and impressions.

Why these are knowable: all five are calculated only from information available by 28 February 2026.

Label / proxy

The March outcome is went_dark. It is set to 1 when a content item has measured GSC days in March but records zero GSC clicks during the March outcome window. This is an observed outcome used as a proxy for a content-review opportunity.

This label does not prove that the page needs a refresh or that a refresh would recover performance.

Context

client_hash_id, content_hash_id, and report_date are identifiers/date fields used for grouping, joining, filtering, and validation. They are not model features.

Excluded

March performance fields such as gsc_clicks and gsc_impressions are excluded from the feature set because they belong to the outcome window. They would reveal information that was not available at the February decision cutoff.

I also exclude future dates, label-derived fields, identifiers as predictive features, and any product-decision fields.

In [44]:
con.execute(f"""
CREATE OR REPLACE VIEW feb_agg AS
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS imp_feb,
    SUM(gsc_clicks) AS clk_feb,

    SUM(gsc_sum_position)
        / NULLIF(SUM(gsc_impressions), 0) AS avg_position_feb,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS days_gsc_feb

FROM read_parquet('{FEB}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING
    SUM(gsc_impressions) >= 100
    AND SUM(gsc_clicks) >= 3
""")

print("✓ February feature aggregate created")

✓ February feature aggregate created


In [45]:
feb_features = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    imp_feb,
    clk_feb,
    avg_position_feb,
    days_gsc_feb,
    100.0 * clk_feb / NULLIF(imp_feb, 0) AS ctr_feb
FROM feb_agg
""").fetchdf()

display(feb_features.head())

print("Feature frame shape:", feb_features.shape)

,client_hash_id,content_hash_id,imp_feb,clk_feb,avg_position_feb,days_gsc_feb,ctr_feb
0,client_3ffa76342f366962,content_32bdebcb01540202,551.0,17.0,3.758621,28,3.085299
1,client_e547b89c05043229,content_4c1e972bec56132e,2882.0,15.0,10.403539,28,0.520472
2,client_e547b89c05043229,content_4e48bd81bb37eb4f,7343.0,3.0,45.407054,28,0.040855
3,client_e547b89c05043229,content_7e131483384291cf,6206.0,6.0,8.934096,28,0.096681
4,client_e547b89c05043229,content_3f12146e57e4baf0,16877.0,29.0,4.443444,28,0.171831


Feature frame shape: (29729, 7)


In [46]:
missing_check = feb_features[
    [
        "imp_feb",
        "clk_feb",
        "avg_position_feb",
        "days_gsc_feb",
        "ctr_feb"
    ]
].isna().mean().mul(100).round(2)

print("Missing percentage by feature:")
display(missing_check.rename("missing_pct"))

Missing percentage by feature:


,missing_pct
imp_feb,0.0
clk_feb,0.0
avg_position_feb,0.0
days_gsc_feb,0.0
ctr_feb,0.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1 — Grain

In [47]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{FEB}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchdf()

display(grain_check)

print("Duplicate combinations:", len(grain_check))

,report_date,client_hash_id,content_hash_id,row_count


Duplicate combinations: 0


Query 2 — February count + dates

In [48]:
feb_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{FEB}')
""").fetchdf()

display(feb_check)

,row_count,first_date,last_date
0,7355108,2026-02-01,2026-02-28


Query 3 — March availability using IS TRUE

In [49]:
mar_availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNTIF(gsc_data_available IS TRUE) AS measured_rows,
        COUNTIF(gsc_data_available IS NOT TRUE) AS unavailable_or_missing_rows
    FROM read_parquet('{MAR}')
""").fetchdf()

display(mar_availability)

,total_rows,measured_rows,unavailable_or_missing_rows
0,9841378,3611061.0,6230317.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits

The warehouse has an unbalanced panel because different clients have different amounts of historical data. Some clients have limited usable search or analytics history, so the February-to-March cohort does not represent every client equally.

Another limitation is that the label is based on observed March search performance. It tells me that a content item showed the defined outcome during March; it does not tell me why that happened or whether a future refresh would recover performance.

The GSC availability flag also matters. Rows without measured GSC data are not treated as zero clicks or zero impressions. This prevents missing measurement from being confused with actual poor performance.

The February feature window and March outcome window are separated by the 28 February decision cutoff, which reduces the risk of using future information.

In [50]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.